## 1. Transcripción Automatizada del Audio (ASR)

En esta primera etapa procesamos las retransmisiones en formato audio (`.mp3`) para extraer el texto transcrito junto con sus marcas de tiempo exactas (`start` y `end`).

**Puntos clave del procesamiento:**
* **Modelo:** Se utiliza `faster-whisper` con el *checkpoint* `large-v3` sobre GPU (`cuda` en `float16`) para optimizar la velocidad de inferencia sin perder precisión léxica.
* **Control de alucinaciones:** Se fija `condition_on_previous_text=False` para evitar que el modelo entre en bucles de repetición provocados por el ruido de fondo, cánticos o murmullos del estadio.
* **Búsqueda:** Se emplea `beam_size=5` para priorizar la coherencia global del discurso de los comentaristas.
* **Caché:** Se comprueba la existencia de transcripciones previas para evitar reprocesar archivos JSON ya generados.

In [8]:
import os
import json
import time
from faster_whisper import WhisperModel

# 1. Configuración de directorios
audio_dir = os.path.join("..", "data", "transcripts")
transcript_dir = os.path.join("..", "data", "transcripts")
os.makedirs(transcript_dir, exist_ok=True)

# 2. Carga del modelo)
# device="cuda" y compute_type="float16"
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("Modelo cargado correctamente. Iniciando procesamiento...\n")

# 3. Bucle de transcripción
for file in os.listdir(audio_dir):
    if file.endswith(".mp3"):
        audio_path = os.path.join(audio_dir, file)
        json_file = f"{os.path.splitext(file)[0]}.json"
        json_path = os.path.join(transcript_dir, json_file)
        
        # Sistema de caché: evitamos reprocesar si el JSON ya existe
        if os.path.exists(json_path):
            print(f"Saltando '{file}', ya existe su transcripción.")
            continue
            
        print(f"Transcribiendo: {file}...")
        start_time = time.time()
        
        # beam_size=5 mejora la coherencia del texto a costa de un poco de velocidad
        # condition_on_previous_text=False evita que el modelo entre en bucles de alucinación con el ruido del estadio
        segments, info = model.transcribe(
            audio_path, 
            language="es",
            beam_size=5,
            condition_on_previous_text=False 
        )
        
        print(f"Detectado idioma '{info.language}' con probabilidad {info.language_probability:.2f}")
        
        # faster-whisper devuelve un generador, iteramos para extraer los datos
        transcription_data = []
        for segment in segments:
            transcription_data.append({
                "id": segment.id,
                "start": segment.start,
                "end": segment.end,
                "text": segment.text.strip()
            })
            
        # Guardado estructurado en JSON
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(transcription_data, f, ensure_ascii=False, indent=4)
            
        elapsed_time = time.time() - start_time
        print(f"✔ Transcripción guardada en {json_path} (Tiempo: {elapsed_time:.2f} segundos)\n")

Modelo cargado correctamente. Iniciando procesamiento...

Transcribiendo: España - Arabia Saudí ｜ Grupo H ｜ Fase de grupos.mp3...
Detectado idioma 'es' con probabilidad 1.00
✔ Transcripción guardada en ..\data\transcripts\España - Arabia Saudí ｜ Grupo H ｜ Fase de grupos.json (Tiempo: 354.19 segundos)

Transcribiendo: España - Argentina ｜ Mundial FIFA 2026.mp3...
Detectado idioma 'es' con probabilidad 1.00
✔ Transcripción guardada en ..\data\transcripts\España - Argentina ｜ Mundial FIFA 2026.json (Tiempo: 695.24 segundos)

Transcribiendo: España - Austria ｜ Dieciseisavos de final.mp3...
Detectado idioma 'es' con probabilidad 1.00
✔ Transcripción guardada en ..\data\transcripts\España - Austria ｜ Dieciseisavos de final.json (Tiempo: 397.94 segundos)

Transcribiendo: España - Bélgica ｜ Cuartos de final.mp3...
Detectado idioma 'es' con probabilidad 1.00
✔ Transcripción guardada en ..\data\transcripts\España - Bélgica ｜ Cuartos de final.json (Tiempo: 375.07 segundos)

Transcribiendo: España

## 2. Filtrado y Normalización Temporal de las Transcripciones

Los archivos de audio brutos contienen pausas, publicidad previa, tiempo de descanso o análisis post-partido. Para analizar con precisión la dinámica temporal del juego, este bloque filtra y estructura los datos según los límites efectivos del encuentro.

**Puntos clave del procesamiento:**
* **Metadatos temporales:** Se leen los intervalos de tiempo reales (`1P`, `2P`, `TE`, `PEN`) almacenados en `matches metadata.json`. Han sido generados a través de gemini aportandole los json originales, se ha hecho comprobación manual posterior
* **Emparejamiento flexible:** Se tokenizan las claves de los partidos (ej. `espana_francia`) para asociar dinámicamente cada archivo independientemente del orden de los equipos en el nombre del `.mp3`.
* **Cálculo de tiempo relativo ($t_{rel}$):** Además de conservar los *timestamps* absolutos del archivo, se calcula el tiempo en segundos desde el inicio exacto de cada parte ($t_{rel} = t_{abs} - t_{inicio\_periodo}$). Esto facilita el alineamiento minuto a minuto para el modelado posterior de series temporales.
* **Almacenamiento estructurado:** Los datos limpios se exportan en formato JSON a la carpeta `data/clean_transcripts/`.

In [15]:
import re
import unicodedata

# 1. Configuración de directorios
data_dir = os.path.join("..", "data")
transcripts_dir = os.path.join(data_dir, "transcripts")
clean_dir = os.path.join(data_dir, "clean_transcripts")
os.makedirs(clean_dir, exist_ok=True)

# Ruta del archivo de metadatos
metadata_path = os.path.join(transcripts_dir, "matches metadata.json")

# 2. Carga dinámica de metadatos desde el archivo JSON
with open(metadata_path, "r", encoding="utf-8") as f:
    match_metadata = json.load(f)

# Función para normalizar texto (quita acentos y caracteres especiales)
def normalize_string(text):
    text = unicodedata.normalize('NFD', text).encode('ascii', 'ignore').decode("utf-8")
    return re.sub(r'[^a-zA-Z0-9\s_]', '', text).lower()

# Listar todos los JSON de transcripciones brutas
raw_json_files = [
    f for f in os.listdir(transcripts_dir) 
    if f.endswith(".json") and f != os.path.basename(metadata_path)
]

# 3. Procesamiento y filtrado de las transcripciones
for match_key, intervals in match_metadata.items():
    # Extraemos las palabras clave dividiendo por '_' (ej. ['espana', 'francia'])
    key_terms = [normalize_string(term) for term in match_key.split('_') if term]
    
    # Buscamos un archivo que contenga TODAS las palabras clave de la llave
    matching_file = None
    for f in raw_json_files:
        norm_file_name = normalize_string(f)
        if all(term in norm_file_name for term in key_terms):
            matching_file = f
            break
    
    if not matching_file:
        print(f"No se encontró archivo transcrito para la clave: '{match_key}'")
        continue

    raw_file_path = os.path.join(transcripts_dir, matching_file)
    with open(raw_file_path, "r", encoding="utf-8") as f:
        segments = json.load(f)

    clean_segments = []
    
    for seg in segments:
        seg_start = seg["start"]
        seg_end = seg["end"]
        
        # Evaluar en qué periodo del partido (1P, 2P, TE, PEN) encaja el segmento
        for period, times in intervals.items():
            if times is not None:
                if seg_start >= times["start"] and seg_end <= times["end"]:
                    clean_segments.append({
                        "id": seg["id"],
                        "period": period,
                        "start": seg_start,
                        "end": seg_end,
                        "relative_start": round(seg_start - times["start"], 2),
                        "relative_end": round(seg_end - times["start"], 2),
                        "text": seg["text"]
                    })
                    break 

    # Guardar la transcripción limpia
    output_filename = f"{match_key}_clean.json"
    output_path = os.path.join(clean_dir, output_filename)
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(clean_segments, f, ensure_ascii=False, indent=4)

    print(f"Procesado: {match_key} -> {len(clean_segments)} segmentos guardados en {output_filename}")

Procesado: espana_arabia_saudi -> 2253 segmentos guardados en espana_arabia_saudi_clean.json
Procesado: espana_argentina -> 3827 segmentos guardados en espana_argentina_clean.json
Procesado: espana_austria -> 2241 segmentos guardados en espana_austria_clean.json
Procesado: espana_belgica -> 2466 segmentos guardados en espana_belgica_clean.json
Procesado: espana_cabo_verde -> 2469 segmentos guardados en espana_cabo_verde_clean.json
Procesado: espana_francia -> 2639 segmentos guardados en espana_francia_clean.json
Procesado: espana_portugal -> 2524 segmentos guardados en espana_portugal_clean.json
Procesado: espana_uruguay -> 2618 segmentos guardados en espana_uruguay_clean.json
